# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

Two retrieval pipelines run over the **same chunked corpus** and are compared:

- **Pipeline 1 — Baseline:** dense bi-encoder retrieval. Embed the query, pull the top 10 chunks by cosine similarity, hand the top 3 to the LLM.
- **Pipeline 2 — Re-ranking:** take the same top 10, re-score every `(query, chunk)` pair with a cross-encoder, re-sort, and hand the new top 3 to the LLM.

The corpus is the Catan base-game rulebook (rules + almanac) — dense with exceptions, penalties and role limits, which makes retrieval mistakes easy to see.

## 1. Setup & Environment

Models used:
- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2`
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Answer generation: `Qwen/Qwen2.5-1.5B-Instruct`, run locally with Hugging Face `transformers`

All three models download from Hugging Face the first time they are used and then run on the local CPU. No API key or network service is required.

In [1]:
import os, re, time, textwrap, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pypdf
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# Retrieval settings
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 10   # candidates pulled by the baseline
TOP_N = 3    # chunks actually sent to the LLM

# Chunking settings
CHUNK_CHARS = 700
CHUNK_OVERLAP = 120

# Answer generation (local Hugging Face model)
LOCAL_MODEL = os.environ.get("LOCAL_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
print("Answer model (local):", LOCAL_MODEL)

C:\Users\PARAM MODI\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Answer model (local): Qwen/Qwen2.5-1.5B-Instruct


## 2. Dataset

A single complex PDF is the knowledge base. The notebook downloads it on first run and keeps it in `data/`.

Each page is extracted separately so every chunk can be traced back to a page number.

In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "catan_rules.pdf"
PDF_URL = "https://world-of-board-games.com.sg/docs/Catan-5ed.pdf"

if not PDF_PATH.exists():
    print("Downloading corpus ...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print("PDF size (KB):", PDF_PATH.stat().st_size // 1024)


def load_pages(path):
    reader = pypdf.PdfReader(str(path))
    if reader.is_encrypted:
        reader.decrypt("")
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if text:
            pages.append({"page": i + 1, "text": text})
    return pages


pages = load_pages(PDF_PATH)
print(f"Loaded {len(pages)} non-empty pages")
print(pages[1]["text"][:400])

PDF size (KB): 3575


Loaded 16 non-empty pages
22
Dear Settlers,
To make it as easy as possible for you to start playing Catan®, we use an award-winning rules system, which consists 
of 3 parts. First, if you do not know how to play Catan, please read the Game Overview on page 16 (the back cover). 
Next, read the Game Rules on pages 2-6 (red borders) and start to play the game. If you have questions during the 
game, consult the Catan Almanac 


## 3. Chunking

The pages are split into overlapping character windows. Paragraph boundaries are respected where possible so a rule and its exception tend to stay together, and a small overlap keeps context from being cut mid-sentence. Each chunk records its source page.

In [3]:
def chunk_pages(pages, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    chunks = []
    for item in pages:
        page_no, text = item["page"], item["text"]
        paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
        buf = ""
        for para in paragraphs:
            if len(buf) + len(para) + 1 <= size:
                buf = (buf + " " + para).strip()
            else:
                if buf:
                    chunks.append({"page": page_no, "text": buf})
                if len(para) > size:
                    # a single long paragraph: slide a window across it
                    for start in range(0, len(para), size - overlap):
                        chunks.append({"page": page_no, "text": para[start:start + size]})
                    buf = ""
                else:
                    tail = buf[-overlap:] if buf else ""
                    buf = (tail + " " + para).strip()
        if buf:
            chunks.append({"page": page_no, "text": buf})
    return chunks


chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(t) for t in chunk_texts) // len(chunk_texts)} chars")
print("\nExample chunk:\n", chunk_texts[5])

Total chunks: 100
Average chunk length: 635 chars

Example chunk:
 cut sheets, always push the tiles through from the front, “cut” side (pushing from the back may cause the tiles to rip). Lay out the map as specified in Illustration A (or on the back of this booklet). First, assemble the frame as shown. Second, create Catan by placing the 19 terrain hexes on the table—again as shown. Third, place the circular number tokens on top of the designated terrain hexes. Finally, place your settlements and roads. Africa is spring and we walk five paths toward the sun


## 4. Pipeline 1 — Baseline Retrieval (Bi-Encoder)

The bi-encoder embeds each chunk independently into a vector. Vectors are L2-normalised and stored in a FAISS inner-product index, so an inner-product search is equivalent to cosine similarity. For a query we embed it the same way and pull the `TOP_K` nearest chunks.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

chunk_embeddings = bi_encoder.encode(
    chunk_texts, normalize_embeddings=True, show_progress_bar=True
).astype("float32")

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"Indexed {index.ntotal} chunks, dimension {dim}")


def baseline_retrieve(query, k=TOP_K):
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{"id": int(i), "score": float(s)} for i, s in zip(ids[0], scores[0])]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:  25%|██▌       | 1/4 [00:00<00:02,  1.05it/s]

Batches:  50%|█████     | 2/4 [00:01<00:01,  1.08it/s]

Batches:  75%|███████▌  | 3/4 [00:03<00:01,  1.04s/it]

Batches: 100%|██████████| 4/4 [00:03<00:00,  1.49it/s]

Batches: 100%|██████████| 4/4 [00:03<00:00,  1.27it/s]

Indexed 100 chunks, dimension 384


## 5. Pipeline 2 — Re-Ranking (Cross-Encoder)

The cross-encoder takes the `TOP_K` baseline candidates and reads each `(query, chunk)` pair **together** in one pass, so its score reflects direct interaction between the query and the passage rather than two separate embeddings. We re-sort the 10 candidates by this score and keep the new top 3.

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)


def rerank(query, candidates):
    ids = [c["id"] for c in candidates]
    pairs = [(query, chunk_texts[i]) for i in ids]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(
        ({"id": i, "score": float(s)} for i, s in zip(ids, scores)),
        key=lambda x: x["score"], reverse=True,
    )
    return ranked

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12701.21it/s]

## 6. Prompt & Answer Generation

Both pipelines use the **identical prompt structure** — only the three retrieved chunks differ. The model is told to answer strictly from the passages and to say so when the answer is absent, which is what exposes a weak retrieval: if the critical chunk is missing, the answer is wrong or incomplete.

The model is loaded once (lazily) and reused. Greedy decoding (`do_sample=False`) keeps the output deterministic, so the two pipelines are compared on their retrieved context, not on sampling noise.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

_tokenizer = None
_model = None


def load_llm():
    global _tokenizer, _model
    if _model is None:
        _tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
        _model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=torch.float32)
        _model.eval()
    return _tokenizer, _model


SYSTEM_PROMPT = (
    "You answer board-game rules questions using only the provided context passages. "
    "Quote the specific rule and cite the passage numbers you used. "
    "If the answer is not contained in the passages, say that you cannot find it."
)


def format_context(ids):
    blocks = []
    for rank, i in enumerate(ids, start=1):
        blocks.append(f"[{rank}] (page {chunks[i]['page']}) {chunk_texts[i]}")
    return "\n\n".join(blocks)


def build_user_prompt(query, ids):
    return f"Context passages:\n{format_context(ids)}\n\nQuestion: {query}\n\nAnswer:"


def generate_answer(query, ids, max_new_tokens=256):
    tokenizer, model = load_llm()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(query, ids)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()

## 7. End-to-end check on a straightforward query

A quick sanity run of both pipelines on an easy question, where the baseline is already expected to do well.

In [7]:
def run_both(query):
    base = baseline_retrieve(query, TOP_K)
    base_ids = [c["id"] for c in base]
    reranked = rerank(query, base)
    rr_ids = [c["id"] for c in reranked]
    return base, base_ids, reranked, rr_ids


def show_candidates(title, items):
    print(title)
    for rank, c in enumerate(items, start=1):
        preview = chunk_texts[c["id"]][:90].replace("\n", " ")
        print(f"  {rank:>2}. chunk {c['id']:>3} (p{chunks[c['id']]['page']}) "
              f"score={c['score']:.3f}  {preview}")


demo_query = "What does a player need to do to win the game?"
base, base_ids, reranked, rr_ids = run_both(demo_query)
show_candidates("Baseline top 10:", base)
print()
show_candidates("Re-ranked top 10:", reranked)

print("\n--- Answer 1 (baseline top 3) ---")
print(generate_answer(demo_query, base_ids[:TOP_N]))
print("\n--- Answer 2 (re-ranked top 3) ---")
print(generate_answer(demo_query, rr_ids[:TOP_N]))

Baseline top 10:
   1. chunk  81 (p14) score=0.460  end of the game. Place them face down in front of you. Of course, if you have 1 or 2 unuse
   2. chunk  12 (p4) score=0.436  ess you’re using the Starting Set-Up for Experienced Players, the oldest player goes first
   3. chunk  25 (p5) score=0.396  Victory Point Cards (Orange Frame) Y You must keep victory point cards hidden. You may onl
   4. chunk  26 (p5) score=0.391  r (including you) has 10 points on their turn. © 2015 Catan GmbH © 2015 Catan GmbH = © 201
   5. chunk  24 (p5) score=0.377  bber” above and follow steps 1 and 2. Once played, knight cards remain face up in front of
   6. chunk  38 (p7) score=0.371  ve me 1 brick for 1 ore?” Beth answers, “If you give me 3 ore, I’ll give you a brick.” Coo
   7. chunk   9 (p3) score=0.351  let. Next, read the Game Rules and start to play. And finally, if you have questions durin
   8. chunk  45 (p8) score=0.347  you get nothing! (However, you can always ask players about the number of

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   7%|▋         | 23/338 [00:00<00:03, 103.19it/s]

Loading weights:  10%|█         | 35/338 [00:00<00:02, 105.33it/s]

Loading weights:  14%|█▎        | 46/338 [00:00<00:04, 69.54it/s] 

...

Loading weights:  95%|█████████▍| 321/338 [00:07<00:00, 38.71it/s]

Loading weights:  96%|█████████▌| 325/338 [00:07<00:00, 35.65it/s]

Loading weights: 100%|██████████| 338/338 [00:07<00:00, 43.61it/s]

A player needs to reach 10 victory points on their turn to win the game. This is stated directly in the passage: "The first player to reach (be at) 10 victory points (VPs) on their turn wins the game."

--- Answer 2 (re-ranked top 3) ---


A player needs to reach 10 victory points on their turn to win the game.


## 8. Failure Mode Analysis (Stress Testing)

Each query below is chosen so the critical chunk sits **outside** the baseline top 3 (buried at ranks 4–10, or missed on wording) but the cross-encoder promotes it into the top 3. Categories covered: keyword trap / distractor, exception & corner case, and negation / constraint.

In [8]:
FAILURE_QUERIES = [
    {
        "category": "Keyword trap / distractor",
        "query": "What happens when a seven is rolled?",
        "note": "The baseline is pulled toward passages that only share wording - the number-token "
                "text that says 'there is no 7', and the table of contents - and misses the actual "
                "robber-and-discard procedure.",
    },
    {
        "category": "Exception & corner case",
        "query": "Can I build a settlement right next to another settlement?",
        "note": "The generic 'how to build a settlement' text ranks high; the Distance Rule "
                "restriction that actually answers the question sits further down.",
    },
    {
        "category": "Negation / constraint",
        "query": "Can I give away resource cards to another player as a gift?",
        "note": "The governing rule is a prohibition ('you cannot give away cards'); the bi-encoder "
                "blurs the negative and surfaces generic trading text instead.",
    },
    {
        "category": "Buried specific (ranks 4-10)",
        "query": "How many points do I need to win?",
        "note": "The exact victory-point total appears in one worked example, buried below several "
                "passages that mention points in general.",
    },
]


def analyse(query, note="", category="", make_answers=True):
    base = baseline_retrieve(query, TOP_K)
    base_ids = [c["id"] for c in base]
    reranked = rerank(query, base)
    rr_ids = [c["id"] for c in reranked]

    print("=" * 100)
    if category:
        print(f"[{category}]")
    print("Q:", query)
    if note:
        print("   ", note)
    print("\nBaseline order (rank: chunk, page, score):")
    for rank, c in enumerate(base, start=1):
        mark = "  <= dropped by baseline top-3" if rank > TOP_N else ""
        print(f"  {rank:>2}: chunk {c['id']:>3} p{chunks[c['id']]['page']} {c['score']:.3f}{mark}")

    print("\nRe-ranked order:")
    for rank, c in enumerate(reranked, start=1):
        print(f"  {rank:>2}: chunk {c['id']:>3} p{chunks[c['id']]['page']} {c['score']:.3f}")

    promoted = [i for i in rr_ids[:TOP_N] if i in base_ids[TOP_N:]]
    records = []
    for pid in promoted:
        before = base_ids.index(pid) + 1
        after = rr_ids.index(pid) + 1
        records.append({"query": query, "category": category, "chunk": pid,
                        "page": chunks[pid]["page"],
                        "baseline_rank": before, "rerank_rank": after})
        print(f"\nPROMOTED chunk {pid}: baseline #{before} -> re-rank #{after} (page {chunks[pid]['page']})")
        print("   ", chunk_texts[pid][:240].replace("\n", " "))

    answers = {}
    if make_answers:
        answers["baseline"] = generate_answer(query, base_ids[:TOP_N])
        answers["rerank"] = generate_answer(query, rr_ids[:TOP_N])
        print("\n--- Answer 1 (baseline top 3) ---\n", answers["baseline"])
        print("\n--- Answer 2 (re-ranked top 3) ---\n", answers["rerank"])
    return records


rank_shift_records = []
for item in FAILURE_QUERIES:
    rank_shift_records += analyse(item["query"], item["note"], item["category"])
    print()

[Keyword trap / distractor]
Q: What happens when a seven is rolled?
    The baseline is pulled toward passages that only share wording - the number-token text that says 'there is no 7', and the table of contents - and misses the actual robber-and-discard procedure.

Baseline order (rank: chunk, page, score):
   1: chunk  57 p10 0.433
   2: chunk  53 p10 0.394
   3: chunk  88 p15 0.368
   4: chunk  64 p11 0.365  <= dropped by baseline top-3
   5: chunk  66 p11 0.301  <= dropped by baseline top-3
   6: chunk  23 p5 0.297  <= dropped by baseline top-3
   7: chunk  69 p12 0.296  <= dropped by baseline top-3
   8: chunk  63 p11 0.293  <= dropped by baseline top-3
   9: chunk   9 p3 0.269  <= dropped by baseline top-3
  10: chunk  94 p16 0.265  <= dropped by baseline top-3

Re-ranked order:
   1: chunk  23 p5 -0.482
   2: chunk  63 p11 -1.440
   3: chunk  64 p11 -1.529
   4: chunk  53 p10 -1.684
   5: chunk  57 p10 -2.857
   6: chunk  94 p16 -5.608
   7: chunk  66 p11 -5.857
   8: chunk   9 


--- Answer 1 (baseline top 3) ---
 When a "7" is rolled, the robber token is activated.

--- Answer 2 (re-ranked top 3) ---
 When a "7" is rolled, none of the players receive resources. Instead:

- Each player counts their resource cards.
- Players with more than 7 resource cards must choose and discard half of them.
- Discarded cards are returned to the supply stack.

If a player holds an odd number of cards, they round down (e.g., 9 cards become 4 discarded).

The robber then moves to the next number token or the desert hex, blocking resource production until the robber moves again.



[Exception & corner case]
Q: Can I build a settlement right next to another settlement?
    The generic 'how to build a settlement' text ranks high; the Distance Rule restriction that actually answers the question sits further down.

Baseline order (rank: chunk, page, score):
   1: chunk  95 p16 0.560
   2: chunk  30 p6 0.550
   3: chunk  70 p12 0.548
   4: chunk  33 p6 0.546  <= dropped by baseline top-3
   5: chunk  77 p13 0.539  <= dropped by baseline top-3
   6: chunk  29 p6 0.519  <= dropped by baseline top-3
   7: chunk  19 p5 0.494  <= dropped by baseline top-3
   8: chunk  67 p12 0.485  <= dropped by baseline top-3
   9: chunk  36 p7 0.459  <= dropped by baseline top-3
  10: chunk  17 p4 0.444  <= dropped by baseline top-3

Re-ranked order:
   1: chunk  67 p12 6.568
   2: chunk  30 p6 5.096
   3: chunk  70 p12 4.379
   4: chunk  95 p16 3.443
   5: chunk  36 p7 3.130
   6: chunk  19 p5 3.078
   7: chunk  33 p6 3.031
   8: chunk  77 p13 1.239
   9: chunk  29 p6 0.287
  10: chunk 


--- Answer 1 (baseline top 3) ---
 No, you cannot build a settlement right next to another settlement. According to the passage, "The second settlement can be placed on any unoccupied intersection, as long as the Distance Rule is observed." This means that while settlements can be built close together, they must still adhere to the distance rule specified by the game's mechanics.

--- Answer 2 (re-ranked top 3) ---
 No, you cannot build a settlement directly next to another settlement without following the Distance Rule. According to the passage, "If you have built all 5 of your settlements, you must upgrade 1 of your settlements to a city before you can build another settlement." This means you need to upgrade an existing settlement to a city before you can build another settlement. Once you've done that, the settlement piece becomes available again and can be used to build another settlement.



[Negation / constraint]
Q: Can I give away resource cards to another player as a gift?
    The governing rule is a prohibition ('you cannot give away cards'); the bi-encoder blurs the negative and surfaces generic trading text instead.

Baseline order (rank: chunk, page, score):
   1: chunk  60 p10 0.496
   2: chunk  59 p10 0.463
   3: chunk  14 p4 0.446
   4: chunk  35 p7 0.409  <= dropped by baseline top-3
   5: chunk  37 p7 0.406  <= dropped by baseline top-3
   6: chunk  52 p9 0.398  <= dropped by baseline top-3
   7: chunk  79 p14 0.384  <= dropped by baseline top-3
   8: chunk  46 p8 0.377  <= dropped by baseline top-3
   9: chunk  51 p9 0.357  <= dropped by baseline top-3
  10: chunk  50 p9 0.355  <= dropped by baseline top-3

Re-ranked order:
   1: chunk  37 p7 4.798
   2: chunk  79 p14 4.155
   3: chunk  59 p10 3.329
   4: chunk  14 p4 2.651
   5: chunk  46 p8 0.849
   6: chunk  60 p10 0.574
   7: chunk  52 p9 0.257
   8: chunk  35 p7 -1.832
   9: chunk  50 p9 -2.082
  10: chu


--- Answer 1 (baseline top 3) ---
 No, according to the rule on page 10, "arn, then no player receives any of that resource that turn." This means that once you have announced a type of resource, no other player will receive any of those resources that turn unless they specifically request them. Therefore, giving away resource cards as a gift would not be allowed under the current rules.

--- Answer 2 (re-ranked top 3) ---
 No, according to the passage, "You may not give away cards."



[Buried specific (ranks 4-10)]
Q: How many points do I need to win?
    The exact victory-point total appears in one worked example, buried below several passages that mention points in general.

Baseline order (rank: chunk, page, score):
   1: chunk  81 p14 0.566
   2: chunk  25 p5 0.551
   3: chunk  82 p14 0.548
   4: chunk  39 p7 0.544  <= dropped by baseline top-3
   5: chunk  40 p7 0.512  <= dropped by baseline top-3
   6: chunk  38 p7 0.505  <= dropped by baseline top-3
   7: chunk  26 p5 0.497  <= dropped by baseline top-3
   8: chunk  47 p9 0.494  <= dropped by baseline top-3
   9: chunk  84 p14 0.476  <= dropped by baseline top-3
  10: chunk  85 p14 0.475  <= dropped by baseline top-3

Re-ranked order:
   1: chunk  39 p7 4.311
   2: chunk  82 p14 3.179
   3: chunk  81 p14 2.856
   4: chunk  25 p5 1.800
   5: chunk  38 p7 0.370
   6: chunk  40 p7 -1.030
   7: chunk  47 p9 -1.782
   8: chunk  85 p14 -2.429
   9: chunk  26 p5 -2.742
  10: chunk  84 p14 -3.287

PROMOTED chunk 39: 


--- Answer 1 (baseline top 3) ---
 To win the game, you need to reach 10 victory points on your turn.

--- Answer 2 (re-ranked top 3) ---
 To win the game, you need 10 victory points.



## 9. Rank-Shift Summary

The chunks that the cross-encoder lifted from ranks 4–10 into the top 3. These are exactly the passages the baseline would have withheld from the LLM.

In [9]:
rank_shift_df = pd.DataFrame(rank_shift_records,
                             columns=["query", "category", "chunk", "page",
                                      "baseline_rank", "rerank_rank"])
rank_shift_df

,query,category,chunk,page,baseline_rank,rerank_rank
0,What happens when a seven is rolled?,Keyword trap / distractor,23,5,6,1
1,What happens when a seven is rolled?,Keyword trap / distractor,63,11,8,2
2,What happens when a seven is rolled?,Keyword trap / distractor,64,11,4,3
3,Can I build a settlement right next to another...,Exception & corner case,67,12,8,1
4,Can I give away resource cards to another play...,Negation / constraint,37,7,5,1
5,Can I give away resource cards to another play...,Negation / constraint,79,14,7,2
6,How many points do I need to win?,Buried specific (ranks 4-10),39,7,4,1


## 10. Latency Profiling

Wall-clock time of the baseline retrieval versus the full re-ranking pipeline (baseline retrieval **plus** the cross-encoder pass). Retrieval-only timing is measured — the LLM call is excluded because it is identical for both pipelines. Each query is run several times and averaged.

In [10]:
def time_baseline(query, repeats=5):
    t0 = time.perf_counter()
    for _ in range(repeats):
        baseline_retrieve(query, TOP_K)
    return (time.perf_counter() - t0) / repeats


def time_rerank_pipeline(query, repeats=5):
    t0 = time.perf_counter()
    for _ in range(repeats):
        cand = baseline_retrieve(query, TOP_K)
        rerank(query, cand)
    return (time.perf_counter() - t0) / repeats


latency_rows = []
for item in FAILURE_QUERIES:
    q = item["query"]
    b = time_baseline(q)
    r = time_rerank_pipeline(q)
    latency_rows.append({"query": q[:45] + "...",
                         "baseline_ms": round(b * 1000, 1),
                         "rerank_pipeline_ms": round(r * 1000, 1),
                         "overhead_ms": round((r - b) * 1000, 1),
                         "slowdown_x": round(r / b, 1)})

latency_df = pd.DataFrame(latency_rows)
latency_df

,query,baseline_ms,rerank_pipeline_ms,overhead_ms,slowdown_x
0,What happens when a seven is rolled?...,23.1,774.1,751.0,33.5
1,Can I build a settlement right next to anothe...,24.3,693.3,669.0,28.6
2,Can I give away resource cards to another pla...,27.1,468.4,441.3,17.3
3,How many points do I need to win?...,22.7,497.6,474.8,21.9


## 11. Discussion pointers

The written analysis — architecture comparison, qualitative answer comparison, latency reading, and the scaling question — is in `report/REPORT.md`, backed by the tables produced above.

In short: the cross-encoder is accurate but costs a forward pass per candidate, so it is used only to re-rank a short baseline shortlist, never to score the whole corpus. Running it over every chunk for every query would multiply cost by the corpus size with no index to prune the search.